In [0]:
# Databricks notebook source

# ============================================================
# 0. CONFIG PATH
# ============================================================

dbutils.widgets.text("path", "")

config_path = dbutils.widgets.get("path")


# ============================================================
# 1. IMPORTS
# ============================================================

import json

import pyspark.sql.functions as F

from common_utils.logging import get_logger
from common_utils.generic_functions import safe_cast


# ============================================================
# 2. LOGGER
# ============================================================

logger = get_logger("dim_product_gold")


# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info(
    "Reading Gold configuration from %s",
    config_path
)

with open(config_path, "r") as f:
    config = json.load(f)

logger.info(
    "Gold configuration loaded successfully"
)


# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================

source_config = config["source"]
target_config = config["target"]

source_name = source_config["source_name"]

source_table = source_config["source_table"]

category_source_table = source_config.get(
    "category_source_table"
)

target_table = target_config["target_table"]


logger.info(
    "[%s] Primary source table: %s",
    source_name,
    source_table
)

logger.info(
    "[%s] Category enrichment table: %s",
    source_name,
    category_source_table
)

logger.info(
    "[%s] Target table: %s",
    source_name,
    target_table
)


# ============================================================
# 5. READ SOURCE TABLES
# ============================================================

logger.info(
    "[%s] Reading primary product source: %s",
    source_name,
    source_table
)

order_products_df = spark.read.table(
    source_table
)

logger.info(
    "[%s] Primary product source loaded successfully",
    source_name
)


logger.info(
    "[%s] Reading category enrichment source: %s",
    source_name,
    category_source_table
)

sales_df = spark.read.table(
    category_source_table
)

logger.info(
    "[%s] Category enrichment source loaded successfully",
    source_name
)


# ============================================================
# 6. TRANSFORM PRODUCT
# ============================================================

def transform_product(
    order_products_df,
    sales_df,
    config
):

    logger.info(
        "Transforming dim_product"
    )

    # --------------------------------------------------------
    # 6.1 PRIMARY PRODUCT DATA
    # --------------------------------------------------------

    product_df = (
        order_products_df
        .select(
            F.col("id").alias("product_id"),
            F.col("product_name"),
            F.col("unit")
        )
        .filter(
            F.col("product_id").isNotNull()
        )
    )

    logger.info(
        "Primary product records prepared"
    )

    # --------------------------------------------------------
    # 6.2 PRODUCT CATEGORY ENRICHMENT
    # --------------------------------------------------------

    sales_product_df = (
        sales_df
        .select(
            F.col("product_id"),
            F.col("product_category").alias("category")
        )
        .filter(
            F.col("product_id").isNotNull()
        )
        .filter(
            F.col("product_category").isNotNull()
        )
        .dropDuplicates(
            ["product_id"]
        )
    )

    logger.info(
        "Product category enrichment prepared"
    )

    # --------------------------------------------------------
    # 6.3 JOIN PRODUCT CATEGORY
    # --------------------------------------------------------

    product_df = (
        product_df
        .join(
            sales_product_df,
            on="product_id",
            how="left"
        )
        .withColumn(
            "category",
            F.coalesce(
                F.col("category"),
                F.lit("Unknown")
            )
        )
    )

    # --------------------------------------------------------
    # 6.4 DEDUPLICATE PRODUCT
    # --------------------------------------------------------

    product_df = (
        product_df
        .dropDuplicates(
            ["product_id"]
        )
    )

    logger.info(
        "dim_product transformation completed"
    )

    return product_df


# ============================================================
# 7. CREATE DIM PRODUCT
# ============================================================

logger.info(
    "[%s] Starting dim_product creation",
    source_name
)

dim_product = transform_product(
    order_products_df,
    sales_df,
    config
)


# ============================================================
# 8. VALIDATE DATA
# ============================================================

logger.info(
    "[%s] Validating dim_product",
    source_name
)

record_count = dim_product.count()

logger.info(
    "[%s] dim_product record count: %s",
    source_name,
    record_count
)

if record_count == 0:

    raise ValueError(
        "dim_product contains zero records"
    )


# ------------------------------------------------------------
# 8.1 PRODUCT ID NULL CHECK
# ------------------------------------------------------------

null_product_id_count = (
    dim_product
    .filter(
        F.col("product_id").isNull()
    )
    .count()
)

logger.info(
    "[%s] Null product_id count: %s",
    source_name,
    null_product_id_count
)

if null_product_id_count > 0:

    raise ValueError(
        "dim_product contains null product_id values"
    )


# ------------------------------------------------------------
# 8.2 PRODUCT ID DUPLICATE CHECK
# ------------------------------------------------------------

duplicate_count = (
    dim_product
    .groupBy("product_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


logger.info(
    "[%s] Duplicate product_id groups: %s",
    source_name,
    duplicate_count
)


if duplicate_count > 0:

    raise ValueError(
        "Duplicate product_id detected in dim_product"
    )


# ============================================================
# 9. DISPLAY
# ============================================================

display(
    dim_product
)


# ============================================================
# 10. CREATE GOLD SCHEMA
# ============================================================

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    retaildataplatform.gold
""")


# ============================================================
# 11. WRITE GOLD TABLE
# ============================================================

logger.info(
    "[%s] Writing dim_product to %s",
    source_name,
    target_table
)

(
    dim_product
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        target_table
    )
)


logger.info(
    "[%s] Gold table %s written successfully",
    source_name,
    target_table
)


# ============================================================
# 12. FINAL VALIDATION
# ============================================================

final_count = (
    spark.read
    .table(target_table)
    .count()
)

logger.info(
    "[%s] Final dim_product record count: %s",
    source_name,
    final_count
)

if final_count == 0:

    raise ValueError(
        "Final dim_product table contains zero records"
    )


logger.info(
    "[%s] dim_product pipeline completed successfully",
    source_name
)

In [0]:
%sql
SELECT COUNT(*) 
FROM retaildataplatform.gold.dim_product;

In [0]:
%sql
SELECT COUNT(*) AS missing_products
FROM retaildataplatform.gold.fact_sales f
LEFT JOIN retaildataplatform.gold.dim_product p
    ON f.product_id = p.product_id
WHERE p.product_id IS NULL;